# 11 · LoRA Fine-Tuning

In plain English, **LoRA** (Low-Rank Adaptation) is a clever trick that lets you fine-tune a big model without touching almost any of its original weights. Instead of re-training the whole model — which would need a lot of memory and produce a huge file — LoRA **freezes** the original model and bolts on a few tiny "adapter" matrices that it trains instead. You end up updating **less than 1%** of the parameters, yet you still teach the model your task.

This is the most common way people fine-tune models today. It runs on modest hardware, finishes fast, and produces an adapter file that's only a few megabytes. In this notebook we'll fine-tune a small DistilBERT model to classify sales **leads** as **hot / warm / cold** — entirely on a laptop CPU, no GPU required.

## What you'll learn

- **Full fine-tuning vs LoRA** — what each one actually changes, and a side-by-side comparison of memory, speed, file size, and when to use each.
- **The intuition behind "low-rank"** — how a big weight change can be approximated by **two small matrices** multiplied together.
- **The PEFT library and `LoraConfig`** — what every key argument means: `r` (rank), `lora_alpha` (scaling), `target_modules` (which layers get adapters), `lora_dropout`, and `task_type`.
- **A complete hands-on LoRA fine-tune**: generate the canonical lead dataset, tokenize it, wrap a DistilBERT classifier with `get_peft_model(...)`, and confirm how few parameters you're actually training.
- **Training with `Trainer`** and a plain-English explanation of every important hyperparameter: learning rate, epochs, batch size, gradient accumulation, max sequence length, and warmup.
- **Evaluating** accuracy on a validation set and **saving** the tiny LoRA adapter to disk.

## Why this matters for fine-tuning

Up to now you've loaded models, prepared data, and run the `Trainer`. The missing piece is making fine-tuning **affordable**. Full fine-tuning a large model can need tens of gigabytes of GPU memory and produce multi-gigabyte files — out of reach for most laptops and budgets.

**LoRA is the answer.** It's the technique behind the vast majority of fine-tunes you'll see on the Hugging Face Hub. Master it and you can adapt powerful models on hardware you already own. And it sets up the **next** notebook: **QLoRA** is simply LoRA with one extra trick (quantization) layered on top to shrink memory even further. Learn LoRA well here, and QLoRA becomes a small step.

## Setup

Run the cell below once. The `%pip install` line is **commented out** — uncomment it if you're on Google Colab or a fresh environment. We add **`peft`** (the library that implements LoRA) to the usual stack.

This notebook is built to run on a plain **CPU** using a tiny model, so anyone can follow along. It will run noticeably **faster on a GPU** if you have one, but a GPU is *not* required.

In [ ]:
# Uncomment the next line on Colab or a fresh environment:
# %pip install transformers datasets peft accelerate torch scikit-learn

import torch
import numpy as np
import transformers, datasets, peft

# Pick the best device available: NVIDIA GPU (cuda), Apple Silicon (mps), or CPU.
if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("transformers:", transformers.__version__)
print("datasets:    ", datasets.__version__)
print("peft:        ", peft.__version__)
print("torch:       ", torch.__version__)
print("device:      ", device)
# Note: this notebook is designed to run fine on CPU with the tiny DistilBERT model.
# It will simply be faster if a GPU (cuda) or Apple Silicon (mps) is detected.

## 1. Full fine-tuning vs LoRA — the big idea

When you **fine-tune** a model, you take its existing weights (the millions or billions of learned numbers) and nudge them toward your task. There are two ways to do that:

**Full fine-tuning** updates **every single weight** in the model.
- The optimizer must keep extra bookkeeping numbers for *every* weight, so memory use is huge.
- The saved result is a **complete copy** of the model — often gigabytes.
- It's powerful, but expensive.

**LoRA** takes a different route. It **freezes** the entire original model (none of those weights change) and **adds** a few small extra matrices — the **adapters** — next to certain layers. Only those tiny adapters are trained.
- You update **less than 1%** of the parameters.
- Memory use drops dramatically.
- The saved result is just the **adapters** — usually a few megabytes — not a whole new model.

Think of it like editing a printed book: full fine-tuning **reprints the entire book** for one correction; LoRA adds a few **sticky notes** in the margins. The original book is untouched, and the sticky notes are small and easy to share.

### What does "low-rank" mean? (no scary math)

A model's weights live in big grids of numbers called **matrices**. When you fine-tune, the *change* you make to a weight matrix is itself a big matrix of "adjustments."

The key insight behind LoRA: **that big change doesn't need to be stored as one big matrix.** It can be closely approximated by **two much smaller matrices multiplied together**. "Low-rank" is just the technical name for "a big change captured by two small matrices."

A quick feel for the savings: suppose a layer's weight matrix is `1000 × 1000` = **1,000,000** numbers.
- Storing the full change = 1,000,000 numbers.
- LoRA with rank `r = 8` stores a `1000 × 8` matrix **plus** an `8 × 1000` matrix = `8,000 + 8,000` = **16,000** numbers.

That's about **1.6%** of the size — and in practice it captures most of what the full change would have done. The number `8` here is the **rank** `r`, the single most important LoRA knob. Smaller `r` = fewer trainable numbers (cheaper, but less capacity); larger `r` = more capacity (and more cost).

The cell below shows this arithmetic concretely.

In [ ]:
# A tiny illustration of the parameter savings (just arithmetic, no model yet).
d = 1000        # a weight matrix is d x d
r = 8           # LoRA rank

full_change   = d * d            # full fine-tuning would learn this many numbers
lora_change   = d * r + r * d    # LoRA learns two small matrices instead

print("full-change parameters:", full_change)
print("LoRA parameters (r=8): ", lora_change)
print("LoRA is this fraction of full: {:.2%}".format(lora_change / full_change))
# Expected:
# full-change parameters: 1000000
# LoRA parameters (r=8):  16000
# LoRA is this fraction of full: 1.60%

**What this does:**

- `full_change = d * d` is how many adjustment numbers a *full* fine-tune would learn for one `1000 × 1000` layer.
- `lora_change = d * r + r * d` is the two small LoRA matrices: one `d × r` and one `r × d`.
- The printed fraction shows LoRA touches a tiny slice of the parameters — the whole reason it's so cheap. Real models have many layers, but the same ratio holds.

### Comparison table

| | **Full fine-tuning** | **LoRA** |
|---|---|---|
| **What changes** | All model weights | Only small added adapters (base frozen) |
| **Trainable params** | 100% | Typically **< 1%** |
| **Memory needed** | High (optimizer state for every weight) | Low (state only for adapters) |
| **Training speed** | Slower | Faster |
| **Saved file size** | Whole model (often GBs) | Just adapters (often a few MB) |
| **Swap between tasks** | Need a full copy per task | Keep one base + many tiny adapters |
| **When to use** | You have lots of compute and need maximum control | Almost always — limited hardware, fast iteration, many tasks |

> **Rule of thumb:** start with LoRA. Reach for full fine-tuning only when you have the hardware *and* evidence that LoRA isn't enough.

### ✏️ Exercise

Using the same arithmetic as above, compute the LoRA fraction for `r = 4`, `r = 16`, and `r = 32` on a `1000 × 1000` layer. Which rank gives the smallest trainable footprint? Print all three percentages.

In [ ]:
# Your turn:
d = 1000
for r in (4, 16, 32):
    frac = (d * r + r * d) / (d * d)
    # print(f"r={r}: {frac:.2%}")

## 2. The PEFT library and `LoraConfig`

**PEFT** stands for **Parameter-Efficient Fine-Tuning**. It's a Hugging Face library that implements LoRA (and similar methods) so you don't have to build the adapter matrices yourself. The workflow is short:

1. Describe the LoRA setup in a **`LoraConfig`**.
2. Wrap your normal model with **`get_peft_model(model, config)`**.
3. Train as usual with `Trainer` — but now only the adapters update.

Here are the **key `LoraConfig` arguments** you'll set, in plain English:

- **`r`** — the **rank** (the `8` from our example). How big the two small matrices are. Bigger = more capacity, more cost. Common values: 8, 16, 32.
- **`lora_alpha`** — a **scaling factor** for how strongly the adapter's output is added back into the model. A common practice is to set it to about `2 × r`. Higher alpha = the adapter speaks louder.
- **`target_modules`** — **which layers get adapters.** You don't adapt every layer — usually the attention projection layers. For DistilBERT these are named `"q_lin"` and `"v_lin"` (the query and value projections).
- **`lora_dropout`** — a small **regularization** that randomly drops part of the adapter during training to reduce overfitting (e.g. `0.05`).
- **`task_type`** — tells PEFT what kind of model this is so it wires the adapters correctly. For classification we use `"SEQ_CLS"` (sequence classification).

In [ ]:
from peft import LoraConfig, TaskType

# We define the config now and USE it in section 5 (after the model is loaded).
lora_config = LoraConfig(
    r=8,                          # rank: size of the two small adapter matrices
    lora_alpha=16,               # scaling: ~2 x r is a common starting point
    target_modules=["q_lin", "v_lin"],  # DistilBERT's query & value projection layers
    lora_dropout=0.05,           # light regularization to curb overfitting
    bias="none",                 # don't train bias terms (keeps it lean)
    task_type=TaskType.SEQ_CLS,  # sequence classification (hot/warm/cold)
)

print(lora_config)
# Note: TaskType.SEQ_CLS is the same as the string "SEQ_CLS".

**What this does:**

- `r=8` and `lora_alpha=16` set the adapter **size** and **strength** (alpha is twice the rank here — a typical choice).
- `target_modules=["q_lin", "v_lin"]` tells PEFT to attach adapters specifically to DistilBERT's **query** and **value** attention layers. These names are model-specific — a different architecture uses different names (see the note in section 5).
- `lora_dropout=0.05` randomly drops 5% of adapter activations during training to help generalization.
- `bias="none"` means we don't bother training bias terms — fewer parameters still.
- `task_type=TaskType.SEQ_CLS` matches our classification head. Using the wrong task type is a common bug.

### ✏️ Exercise

Create a **second** `LoraConfig` called `lora_config_big` with `r=16` and `lora_alpha=32`, keeping everything else the same. We won't train with it, but in section 5 you'll be able to compare how many trainable parameters each config produces.

In [ ]:
# Your turn:
# lora_config_big = LoraConfig(
#     r=16,
#     lora_alpha=32,
#     target_modules=["q_lin", "v_lin"],
#     lora_dropout=0.05,
#     bias="none",
#     task_type=TaskType.SEQ_CLS,
# )

## 3. The lead dataset

Our task: predict a sales lead's **intent** — `hot`, `warm`, or `cold` — from a few features (family size, income, whether they rent or own, what call-to-action they clicked, etc.). This is the **canonical dataset** used across this course, so you'll recognize it.

We generate 200 synthetic leads with a fixed random seed so everyone gets the **same** data. Each lead gets a `lead_intent` label computed from a simple scoring rule.

In [ ]:
import random
random.seed(42)
MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
CTAS = ["requested_quote","booked_demo","downloaded_brochure","newsletter_signup"]
CONDITIONS = ["urgent","exploring","just_browsing"]

def make_lead():
    family_size = random.randint(1, 6)
    income = random.choice([25000,40000,55000,70000,85000,100000,120000,150000])
    rent_or_own = random.choice(["rent","own"])
    cta = random.choice(CTAS)
    engagement_month = random.choice(MONTHS)
    current_condition = random.choice(CONDITIONS)
    score = 0
    if cta in ("requested_quote","booked_demo"): score += 2
    elif cta == "downloaded_brochure": score += 1
    if rent_or_own == "own": score += 1
    if income >= 80000: score += 1
    if family_size >= 4: score += 1
    if current_condition == "urgent": score += 2
    elif current_condition == "exploring": score += 1
    lead_intent = "hot" if score >= 5 else ("warm" if score >= 3 else "cold")
    return {"family_size": family_size, "income": income, "rent_or_own": rent_or_own,
            "cta": cta, "engagement_month": engagement_month,
            "current_condition": current_condition, "lead_intent": lead_intent}

leads = [make_lead() for _ in range(200)]

print("total leads:", len(leads))
print("first lead :", leads[0])

# Quick look at the label balance:
from collections import Counter
print("label counts:", Counter(l["lead_intent"] for l in leads))

**What this does:**

- `random.seed(42)` makes the generation **reproducible** — you'll get the exact same 200 leads every run.
- `make_lead()` builds one lead with random features, then applies a transparent **scoring rule** to assign `hot` / `warm` / `cold`.
- `Counter(...)` shows the **class balance**. Real datasets are often imbalanced; knowing the balance helps you read the accuracy later (e.g. if 60% are "warm", a lazy model could score 60% by always guessing "warm").

### Turning each lead into text and a numeric label

Our model reads **text**, and the `Trainer` needs an integer **label**. So for each lead we build:

- a short **text** string describing the lead, and
- a **label** integer using the map `{"cold": 0, "warm": 1, "hot": 2}`.

In [ ]:
LABEL2ID = {"cold": 0, "warm": 1, "hot": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def lead_to_text(lead):
    return (f"family size {lead['family_size']}, income {lead['income']}, "
            f"{lead['rent_or_own']} home, CTA {lead['cta']}, "
            f"month {lead['engagement_month']}, condition {lead['current_condition']}")

texts  = [lead_to_text(l) for l in leads]
labels = [LABEL2ID[l["lead_intent"]] for l in leads]

print("example text :", texts[0])
print("example label:", labels[0], "->", ID2LABEL[labels[0]])

**What this does:**

- `LABEL2ID` maps each class name to an integer the model can predict; `ID2LABEL` is the reverse, for printing readable names later.
- `lead_to_text(lead)` flattens a lead's fields into one short sentence. The model never sees the raw dict — only this text.
- `texts` and `labels` are now two parallel lists: the input strings and their correct class ids.

### ✏️ Exercise

Print the **text and label** for `leads[5]` and `leads[10]`. Do the labels match your intuition given the income, CTA, and condition? (Remember the scoring rule: urgent + a quote/demo + higher income push toward "hot".)

In [ ]:
# Your turn:
# for i in (5, 10):
#     print(texts[i], "=>", ID2LABEL[labels[i]])

## 4. Tokenize, build a `Dataset`, and split

Now we turn the text into numbers the model can read, package everything as a `datasets.Dataset`, and carve out a **validation** set so we can measure real learning (not memorization).

We use the tokenizer that matches our base model, **`distilbert-base-uncased`**.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 64   # max sequence length: pad/truncate every example to 64 tokens

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Build a Dataset from our parallel lists. Column names matter:
# the Trainer expects the target column to be called "label".
raw_ds = Dataset.from_dict({"text": texts, "label": labels})

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",   # pad short sequences up to MAX_LEN
        truncation=True,        # cut long sequences down to MAX_LEN
        max_length=MAX_LEN,
    )

tokenized_ds = raw_ds.map(tokenize_batch, batched=True)
print("columns:", tokenized_ds.column_names)
print("one row input_ids length:", len(tokenized_ds[0]["input_ids"]))  # -> 64

**What this does:**

- `AutoTokenizer.from_pretrained(MODEL_NAME)` loads the **exact** tokenizer DistilBERT was trained with.
- `Dataset.from_dict({"text": ..., "label": ...})` builds the dataset. The label column **must** be named `label` (or `labels`) for the `Trainer` to find it.
- `tokenize_batch` pads/truncates every example to `MAX_LEN=64` tokens so they form neat rectangular batches. Our lead sentences are short, so 64 is plenty.
- `.map(..., batched=True)` adds `input_ids` and `attention_mask` columns efficiently.

In [ ]:
# Hold out 20% for validation so we can measure generalization.
split = tokenized_ds.train_test_split(test_size=0.2, seed=42)
train_ds = split["train"]
val_ds   = split["test"]

print("train rows:", len(train_ds))   # -> 160
print("val rows:  ", len(val_ds))     # -> 40

**What this does:**

- `train_test_split(test_size=0.2, seed=42)` reserves 20% of rows (40 leads) for **validation** and keeps 80% (160 leads) for training.
- `seed=42` makes the split reproducible.
- We never train on `val_ds`; it's our honest check of whether the model learned the pattern or just memorized examples.

### ✏️ Exercise

Re-run the split with `test_size=0.3` and print the new train/validation sizes. With more data held out for validation, how many examples remain for training?

In [ ]:
# Your turn:
# split_30 = tokenized_ds.train_test_split(test_size=0.3, seed=42)
# print("train:", len(split_30["train"]), "val:", len(split_30["test"]))

## 5. Load the classifier and wrap it with LoRA

Now the heart of the notebook. We:

1. Load **`distilbert-base-uncased`** as a sequence classifier with **`num_labels=3`** (hot/warm/cold).
2. Wrap it with **`get_peft_model(model, lora_config)`** — this freezes the base model and inserts the LoRA adapters.
3. Call **`print_trainable_parameters()`** to see how few parameters we're actually training.

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import get_peft_model

# Load the base model with a fresh 3-class classification head.
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,   # so predictions can show names, not just ids
    label2id=LABEL2ID,
)
# (You'll see a warning that the classifier head is newly initialized -- that's
#  expected: those weights are brand new and WILL be trained by LoRA.)

# Wrap the base model with our LoRA config: freezes the base, adds adapters.
model = get_peft_model(base_model, lora_config)

# The headline result: how much are we actually training?
model.print_trainable_parameters()
# Expected (numbers approximate):
# trainable params: ~150K || all params: ~67M || trainable%: ~0.2-1.0%

**What this does:**

- `AutoModelForSequenceClassification.from_pretrained(..., num_labels=3)` loads DistilBERT and attaches a **new** 3-way classification head. The warning about "newly initialized" weights is normal — that head starts random and gets trained.
- `get_peft_model(base_model, lora_config)` is the LoRA magic: it **freezes** all ~67M base parameters and adds the small adapter matrices on the `q_lin`/`v_lin` layers.
- `print_trainable_parameters()` prints something like **trainable% ≈ 0.2–1%**. That tiny percentage is the whole point: you're teaching the model your task while leaving 99%+ of it frozen. The classifier head is also trainable, which is why the count isn't *only* the adapters.

> **On a GPU you'd scale up.** This runnable example uses tiny DistilBERT as a **classifier** so it works on CPU. For a real GPU run you'd often swap in a larger **generative** base like `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, use `target_modules=["q_proj", "v_proj"]` (that model's attention layer names), and set `task_type="CAUSAL_LM"` instead of `"SEQ_CLS"`. The **PEFT workflow is identical** — only the base model, the target-module names, and the task type change.

### ✏️ Exercise

If you created `lora_config_big` (r=16) in section 2's exercise, wrap a *fresh* copy of the base model with it and call `print_trainable_parameters()`. How does the trainable count compare to `r=8`? (Hint: reload the base model first so you don't double-wrap.)

In [ ]:
# Your turn (uncomment if you defined lora_config_big):
# fresh_base = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)
# big_model = get_peft_model(fresh_base, lora_config_big)
# big_model.print_trainable_parameters()

## 6. Training hyperparameters — what every knob means

Before we hit "train," let's understand the settings. These go into `TrainingArguments`. Getting a feel for them is one of the most valuable fine-tuning skills.

- **`learning_rate`** — how big a step the optimizer takes each update. Too high and training is unstable; too low and it barely learns. **LoRA tolerates a higher learning rate** than full fine-tuning (the base is frozen), so values like `2e-4` are common — higher than the `2e-5`-ish you'd use for a full fine-tune.
- **`num_train_epochs`** — how many full passes over the training data. More epochs = more learning, but too many can **overfit** (memorize) the training set. We use a small number (3–5) for a quick CPU run.
- **`per_device_train_batch_size`** — how many examples the model processes at once before each weight update. Bigger batches are steadier but use more memory. On CPU we keep it small (8).
- **`gradient_accumulation_steps`** — a trick to **simulate a bigger batch** without the memory cost. With batch size 8 and accumulation 2, the model adds up gradients over 2 mini-batches before updating — an **effective batch size of 16**. Great when memory is tight.
- **max sequence length** (`max_length` in the tokenizer, set in section 4) — the longest input in tokens. Longer = more memory and slower. We used 64 because our lead sentences are short.
- **`warmup_steps` / `warmup_ratio`** — start the learning rate near zero and ramp it up over the first few steps. This **warmup** avoids a big, destabilizing first jump. `warmup_ratio=0.1` means "warm up over the first 10% of training."

> **Quick analogy:** the *learning rate* is your stride length, *epochs* are how many laps you run, *batch size* is how many problems you grade before adjusting your teaching, *gradient accumulation* is grading several small stacks before adjusting (to act like one big stack), and *warmup* is jogging slowly before you sprint.

## 7. Define the accuracy metric

We'll measure **accuracy** — the fraction of validation leads the model labels correctly. The `Trainer` calls a `compute_metrics` function after each evaluation, handing us the model's predictions and the true labels.

> Accuracy is a great starting metric. **Notebook 13** covers richer metrics — **precision, recall, and F1** — which matter when classes are imbalanced.

In [ ]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, true_labels = eval_pred          # logits: raw scores per class
    predictions = np.argmax(logits, axis=-1) # pick the highest-scoring class
    acc = accuracy_score(true_labels, predictions)
    return {"accuracy": acc}

# Quick self-test of the function with fake data:
fake = (np.array([[2.0, 0.1, 0.1], [0.1, 0.1, 3.0]]), np.array([0, 2]))
print(compute_metrics(fake))   # -> {'accuracy': 1.0} (both predicted correctly)

**What this does:**

- `eval_pred` is a tuple of `(logits, true_labels)` the `Trainer` passes in.
- `np.argmax(logits, axis=-1)` turns each row of raw class scores into a single predicted class id (the highest one).
- `accuracy_score(true, pred)` from scikit-learn computes the fraction correct. We return it in a dict so the `Trainer` logs it as `eval_accuracy`.

### Set up `TrainingArguments` and `Trainer`, then train

Now we plug in the hyperparameters from section 6 and run the LoRA fine-tune. Only the adapters (and the classifier head) update — the base DistilBERT stays frozen.

> Heads up: this trains a real model on CPU. With 160 examples and a few epochs it typically takes **a couple of minutes**. That's expected.

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="lora_lead_classifier",   # where checkpoints/logs go
    learning_rate=2e-4,                   # higher is fine for LoRA (base is frozen)
    num_train_epochs=4,                   # a few passes over the data (CPU-friendly)
    per_device_train_batch_size=8,        # 8 examples per step
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,        # accumulate 2 steps -> effective batch 16
    warmup_ratio=0.1,                     # ramp LR up over the first 10% of steps
    weight_decay=0.01,                    # mild regularization on the weights
    logging_steps=10,
    eval_strategy="epoch",                # evaluate once per epoch
    save_strategy="no",                   # don't write checkpoints (keep it light)
    report_to="none",                     # no external logging
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()   # <-- runs the LoRA fine-tune. A couple of minutes on CPU.
print("LoRA fine-tuning finished!")

**What this does:**

- `TrainingArguments(...)` collects the knobs from section 6: `learning_rate=2e-4` (LoRA-friendly), `num_train_epochs=4`, `per_device_train_batch_size=8`, `gradient_accumulation_steps=2` (effective batch 16), and `warmup_ratio=0.1`.
- `eval_strategy="epoch"` runs our `compute_metrics` on the validation set after **every epoch**, so you can watch accuracy climb.
- `save_strategy="no"` skips intermediate checkpoints to keep the run light (we save the final adapter ourselves in section 9).
- `trainer.train()` runs the whole loop — but thanks to LoRA, the gradients and optimizer state only cover the **tiny** adapter (plus head), so it's cheap. Watch the **loss** fall and **eval_accuracy** rise.

> If an older `transformers` version complains about `eval_strategy`, use the older name `evaluation_strategy="epoch"` instead.

## 8. Evaluate on the validation set

Training loss going down is encouraging, but the honest test is **validation accuracy** — performance on leads the model never trained on.

In [ ]:
metrics = trainer.evaluate()
print(metrics)
# Expect something like:
# {'eval_loss': 0.5..., 'eval_accuracy': 0.8..., 'eval_runtime': ...}
print("Validation accuracy: {:.1%}".format(metrics["eval_accuracy"]))

**What this does:**

- `trainer.evaluate()` runs the model over `val_ds` and returns a dict including `eval_loss` and our `eval_accuracy`.
- A good result here means LoRA successfully taught DistilBERT the lead-intent pattern while training **less than 1%** of the model. Exact numbers vary run to run; with this small synthetic dataset you should see accuracy well above random guessing (random = ~33% for 3 classes).
- If accuracy is low, see the **Common mistakes** section — usually it's too few epochs, a learning rate that's too small, or a label/column-name mismatch.

In [ ]:
# See it in action: predict on a few validation leads and compare to the truth.
import torch

model.eval()
sample = val_ds.select(range(5))   # first 5 validation rows
inputs = {
    "input_ids": torch.tensor(sample["input_ids"]),
    "attention_mask": torch.tensor(sample["attention_mask"]),
}
with torch.no_grad():
    logits = model(**inputs).logits
preds = torch.argmax(logits, dim=-1).tolist()

for true_id, pred_id in zip(sample["label"], preds):
    mark = "OK " if true_id == pred_id else "X  "
    print(f"{mark} true={ID2LABEL[true_id]:5s}  predicted={ID2LABEL[pred_id]}")

**What this does:** we grab 5 validation rows, build a tensor batch, and run a forward pass with `torch.no_grad()` (no training, saves memory). `argmax` over the logits gives the predicted class id, which `ID2LABEL` turns back into `hot`/`warm`/`cold`. Lining up `true` vs `predicted` makes the model's behavior concrete — you can literally see where it's right and wrong.

### ✏️ Exercise

Increase `num_train_epochs` from 4 to 6 in the `TrainingArguments` cell, re-run training and `trainer.evaluate()`. Did validation accuracy improve, stay flat, or drop? (If it drops while training loss keeps falling, that's a sign of **overfitting**.)

## 9. Save the LoRA adapter (it's tiny!)

The payoff: instead of saving a whole new model, we save **just the adapter**. `save_pretrained` on a PEFT model writes only the small adapter weights plus a config that records which base model they belong to.

In [ ]:
import os

adapter_dir = "lora_lead_adapter"
model.save_pretrained(adapter_dir)          # saves ONLY the LoRA adapter
tokenizer.save_pretrained(adapter_dir)      # keep the tokenizer alongside it

print("saved files:", os.listdir(adapter_dir))
# You'll see adapter_model.safetensors, adapter_config.json, plus tokenizer files.

# How small is the adapter? Sum the saved file sizes:
total_bytes = sum(
    os.path.getsize(os.path.join(adapter_dir, f)) for f in os.listdir(adapter_dir)
)
print("adapter folder size: {:.2f} MB".format(total_bytes / 1e6))
# Typically just a few MB -- vs hundreds of MB for the full DistilBERT model.

**What this does:**

- `model.save_pretrained(adapter_dir)` writes `adapter_model.safetensors` (the trained adapter weights) and `adapter_config.json` (which base model + LoRA settings). It does **not** copy the big base model.
- Saving the **tokenizer** alongside keeps everything you need to reload in one place.
- The printed folder size shows why LoRA is so practical: a few **megabytes** instead of the full model's size. You can keep one base model plus many small adapters for different tasks.

### Loading the adapter back (for reference)

To use the adapter later, you load the **base** model and then apply the adapter on top with `PeftModel.from_pretrained`:

```python
from transformers import AutoModelForSequenceClassification
from peft import PeftModel

base = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=3,
    id2label=ID2LABEL, label2id=LABEL2ID,
)
loaded = PeftModel.from_pretrained(base, "lora_lead_adapter")
# 'loaded' now behaves like your fine-tuned model.
```

That's the whole round trip: train a tiny adapter, save it, reload it on top of the original base.

### ✏️ Exercise

Save the adapter to a **second** folder named `"lead_adapter_backup"`, then (using the reference snippet above) load the base model and apply that adapter. Run one prediction on `texts[0]` to confirm the reloaded model works.

In [ ]:
# Your turn:
# model.save_pretrained("lead_adapter_backup")
# ... then load base + PeftModel.from_pretrained(base, "lead_adapter_backup")

## Common mistakes & how to debug them

- **Wrong `target_modules` names.** Layer names are **architecture-specific**. DistilBERT uses `"q_lin"`/`"v_lin"`; many LLaMA-style models use `"q_proj"`/`"v_proj"`. If `get_peft_model` trains **0** adapter params or errors, your names are wrong — print `model` (or `base_model`) to inspect the actual layer names.
- **Wrong `task_type`.** Use `"SEQ_CLS"` for classification and `"CAUSAL_LM"` for text generation. A mismatch leads to shape errors or the adapters being wired to the wrong place.
- **Label column not named `label`.** The `Trainer` looks for `label` (or `labels`). If you named it `lead_intent`, you'll get a `KeyError` or it will silently ignore your targets.
- **Learning rate too low.** LoRA wants a **higher** LR than full fine-tuning. If loss barely moves, try `2e-4` or even `3e-4` instead of `2e-5`.
- **Forgetting to wrap the model.** If you train `base_model` directly (not the `get_peft_model(...)` result), you're doing a **full** fine-tune, not LoRA — slower and heavier. Always train the wrapped `model`.
- **Saving the wrong thing.** Calling `save_pretrained` on the **PEFT** model saves the small adapter. If you expected a few MB but got hundreds, you saved a full model — check you're saving the wrapped `model`.
- **`eval_strategy` vs `evaluation_strategy`.** Newer `transformers` uses `eval_strategy`; older versions use `evaluation_strategy`. If one errors, switch to the other.
- **Out of memory (on GPU).** Lower `per_device_train_batch_size`, raise `gradient_accumulation_steps` to keep the effective batch, or shorten `max_length`.

## Summary

- **Full fine-tuning** updates every weight (heavy, big files); **LoRA freezes the base** and trains tiny added **adapter** matrices, touching **< 1%** of parameters.
- **"Low-rank"** just means a big weight change is approximated by **two small matrices** multiplied together — `r` controls their size and thus the trainable footprint.
- The **PEFT** library implements this. You describe it in a **`LoraConfig`** (`r`, `lora_alpha`, `target_modules`, `lora_dropout`, `task_type`) and apply it with **`get_peft_model(model, config)`**.
- We fine-tuned **`distilbert-base-uncased`** as a 3-class lead classifier on the canonical dataset, confirmed the **tiny trainable %** with `print_trainable_parameters()`, and trained with **`Trainer`** — understanding every hyperparameter (learning rate, epochs, batch size, gradient accumulation, max length, warmup).
- We **evaluated accuracy** on a held-out validation set (richer metrics come in notebook 13) and **saved just the adapter** — a few MB you can reload on top of the base model anytime.
- **LoRA is the standard, practical way to fine-tune today.** You can now adapt real models on modest hardware.

## What to learn next

Next up: **`12_qlora_finetuning.ipynb`**. QLoRA takes everything you just learned and adds one more memory-saving trick — **quantization** — which compresses the frozen base model into a much smaller numeric format (like 4-bit) before attaching the LoRA adapters. The result: you can fine-tune **much larger** models on the same hardware. Since QLoRA is "LoRA + quantization," the workflow will feel familiar — you've already done the hard part here.